# Metacognition with a vision-language model

Qwen3-VL-2B-Instruct says which of two boxes has more dots, then how sure it is. The displays
are drawn by `stimuli.py` in this folder; the two questions are asked by `observer.py`.

- **Dots:** 313 in one box (half of 625 places), 313 + the difference in the other, as in
  Rouault et al. (2018)
- **Differences and trials:** set in `stimuli.py` (`DIFFERENCES`, `PAIRS`); each pair is one
  display and the same display with the boxes swapped

Results go to `../data/model_results.csv`, one row per display. `analysis.ipynb` makes the
numbers and figures from that file.

Runs on a GPU (Colab: Runtime > Change runtime type > T4 GPU) or, more slowly, on a laptop CPU.

## Setup

In [1]:
import os
import sys

inColab = "google.colab" in sys.modules

if inColab:
  from google.colab import drive
  drive.mount("/content/drive")
  os.chdir("/content/drive/MyDrive/Projects/vlm-psychophysics/chapters/metacognition/code")
  os.system("pip install -q -U transformers")

sys.path.insert(0, os.getcwd())
print("running in Colab:", inColab)

Mounted at /content/drive
running in Colab: True


In [2]:
import time
import torch
import numpy  as np
import pandas as pd
from transformers.utils import logging

import observer
import stimuli

logging.disable_progress_bar()
logging.set_verbosity_error()

device = "cuda" if torch.cuda.is_available() else "cpu"
obs    = observer.Observer(device=device)
print("device:", device)
print("differences:", stimuli.DIFFERENCES, "| pairs per difference:", stimuli.PAIRS)

device: cpu
differences: [5, 10, 20, 40, 70] | pairs per difference: 30


## The two questions

The chat as the model gets it, after it has answered "Left". The image is one block of
`<|image_pad|>` tokens.

In [3]:
chat = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": observer.QUESTION}]},
        {"role": "assistant", "content": [{"type": "text", "text": "Left"}]},
        {"role": "user", "content": [{"type": "text", "text": observer.CONFIDENCE}]}]
print(obs.proc.apply_chat_template(chat, tokenize=False, add_generation_prompt=True))

<|im_start|>user
<|vision_start|><|image_pad|><|vision_end|>Which box has more dots: left or right? Answer with one word.<|im_end|>
<|im_start|>assistant
Left<|im_end|>
<|im_start|>user
How sure are you that your answer is correct? Answer with one number from 1 (guessing) to 5 (certain).<|im_end|>
<|im_start|>assistant



## The scoring

- **Answer:** "left" or "right", whichever the model is more likely to write first.
- **Hidden confidence:** the probability of that answer, out of "left" + "right" (50% to 100%).
- **Stated confidence:** after its answer, the digit 1 to 5 it is most likely to write.
- A display is **correct** if the answer names the box with more dots.

First, the model's lean to "left" when there is nothing to tell the boxes apart.

In [4]:
grey  = np.full((stimuli.CANVAS_PX[1], stimuli.CANVAS_PX[0]), stimuli.GREY, np.uint8)
equal = [stimuli.draw_trial(dict(difference=0, fuller="left", seed=s)) for s in range(8)]
for name, greys in [("grey image", [grey]), ("313 vs 313 dots, 8 displays", equal)]:
  d = pd.DataFrame(obs.ask(greys))
  print("%-28s P(left) out of left + right: %.2f | stated: %s"
        % (name, (d.p_left / (d.p_left + d.p_right)).mean(), d.stated.tolist()))

grey image                   P(left) out of left + right: 0.61 | stated: [5]
313 vs 313 dots, 8 displays  P(left) out of left + right: 0.57 | stated: [5, 5, 5, 5, 5, 5, 5, 5]


## The run

Each pair goes through the model as one batch (two displays). If the run stops, run the cell
again: displays already in the file are skipped.

In [5]:
resultsFile = "../data/model_results.csv"

results = pd.read_csv(resultsFile).to_dict("records") if os.path.exists(resultsFile) else []
done    = set(r["index"] for r in results)
todo    = [t for t in stimuli.make_trials() if t["index"] not in done]
print("%d displays done, %d to go" % (len(done), len(todo)))

startTime = time.time()
for b in range(0, len(todo), 2):
  batch = todo[b:b + 2]
  for trial, answer in zip(batch, obs.ask([stimuli.draw_trial(t) for t in batch])):
    results.append({**trial, **answer, "model": observer.MODEL, "correct": int(answer["answer"] == trial["fuller"])})
  if (b // 2) % 20 == 19:
    pd.DataFrame(results).to_csv(resultsFile, index=False)
    print("%d displays, %.0f s" % (len(results), time.time() - startTime), flush=True)

pd.DataFrame(results).to_csv(resultsFile, index=False)
print("done: %d displays in %.0f s" % (len(results), time.time() - startTime))

300 displays done, 0 to go
done: 300 displays in 0 s


In [6]:
df = pd.DataFrame(results)
print(df.groupby("difference").agg(correct=("correct", "mean"), hidden=("hidden", "mean"),
                                   stated=("stated", "mean")).round(2).to_string())

            correct  hidden  stated
difference                         
5              0.58    0.70     5.0
10             0.52    0.73     5.0
20             0.58    0.70     5.0
40             0.70    0.69     5.0
70             0.77    0.71     5.0
